# 04 — Causal self-attention in GPT-2 small: 12 heads × 64 dimensions

GPT-2 small's attention, per block:

| piece | shape | parameters |
|---|---|---|
| `c_attn` (fused q, k, v projection) | 768 → 2304 | 1,769,472 + 2,304 = 1,771,776 |
| `c_proj` (output projection) | 768 → 768 | 589,824 + 768 = 590,592 |
| **one attention layer** | | **2,362,368** |
| **12 layers** | | **28,348,416** |

Each of the 12 heads works with a 64-dimensional slice of the 768 features, and the scores are scaled by $1/\sqrt{64} = 1/8$.

Attention is the only layer in GPT that mixes information **across positions**, so it's the only backward pass where the gradient at one position depends on other positions. We build it in three steps:

1. one head, one sequence: derive every gradient by hand;
2. why the $1/\sqrt{hs}$ scale matters *for gradients*;
3. many heads and a batch: the reshape/transpose bookkeeping, the fused `c_attn` projection, and a full gradient check.

We also find a shape bug in your script's `MultiHeadAttention`.

In [ ]:
import math
import numpy as np
np.set_printoptions(precision=4, suppress=True, linewidth=120)

def numerical_grad(f, x, eps=1e-6):
    grad = np.zeros_like(x, dtype=np.float64)
    it = np.nditer(x, flags=["multi_index"])
    while not it.finished:
        i = it.multi_index
        old = x[i]
        x[i] = old + eps; fp = f()
        x[i] = old - eps; fm = f()
        x[i] = old
        grad[i] = (fp - fm) / (2 * eps)
        it.iternext()
    return grad

def rel_error(a, b):
    return np.linalg.norm(a - b) / max(1e-12, np.linalg.norm(a) + np.linalg.norm(b))

def spot_check(f, p, g, k=20, eps=1e-5, seed=123):
    """Check k random entries of the analytic gradient g (of scalar f() w.r.t. array p) against
    central differences. Entries with near-zero gradient are skipped: there the finite difference
    is pure rounding noise, so a relative error would be meaningless."""
    r = np.random.default_rng(seed)
    candidates = np.flatnonzero(np.abs(g) > 1e-3 * np.abs(g).max())
    worst = 0.0
    for i in r.choice(candidates, size=min(k, candidates.size), replace=False):
        ii = np.unravel_index(i, p.shape)
        old = p[ii]
        p[ii] = old + eps; fp = f()
        p[ii] = old - eps; fm = f()
        p[ii] = old
        num = (fp - fm) / (2 * eps)
        worst = max(worst, abs(num - g[ii]) / (abs(num) + abs(g[ii])))
    return worst

class Softmax:                                  # from notebook 02
    def forward(self, x):
        z = x - x.max(axis=-1, keepdims=True)
        e = np.exp(z)
        self.s = e / e.sum(axis=-1, keepdims=True)
        return self.s
    def backward(self, dout):
        s = self.s
        return s * (dout - (dout * s).sum(axis=-1, keepdims=True))

class Linear:                                   # from notebook 01
    def __init__(self, in_f, out_f, bias=True, std=0.02, rng=None, dtype=np.float64):
        rng = rng if rng is not None else np.random.default_rng()
        self.params = {"W": rng.normal(0.0, std, (in_f, out_f)).astype(dtype)}
        if bias:
            self.params["b"] = np.zeros(out_f, dtype=dtype)
        self.grads = {k: np.zeros_like(v) for k, v in self.params.items()}
    def forward(self, x):
        self.x = x
        out = x @ self.params["W"]
        return out + self.params["b"] if "b" in self.params else out
    def backward(self, dout):
        x2 = self.x.reshape(-1, self.x.shape[-1])
        d2 = dout.reshape(-1, dout.shape[-1])
        self.grads["W"] += x2.T @ d2
        if "b" in self.params:
            self.grads["b"] += d2.sum(axis=0)
        return dout @ self.params["W"].T
    def parameters(self, prefix=""):
        return [(prefix + k, self.params[k], self.grads[k]) for k in self.params]

rng = np.random.default_rng(0)

## 1. One head, one sequence

Let $Q, K, V \in \mathbb{R}^{T\times d}$, where $d$ is the head size `hs` (64 in GPT-2 small). Then

$$
S = \frac{QK^\top}{\sqrt d}\;(T\times T),\qquad
\tilde S_{ij} = \begin{cases} S_{ij} & j\le i\\ -\infty & j>i\end{cases},\qquad
A = \operatorname{softmax}_\text{rows}(\tilde S),\qquad
Y = AV\;(T\times d).
$$

Row $i$ of $A$ says how much query position $i$ reads from each key position $j\le i$. Row $i$ of $Y$ is the resulting weighted average of value vectors.

### The one new tool: a product of two *variables*

In Linear, one factor was a parameter. Here both factors depend on the input. The rule is the same, applied to each factor:

$$
Z = PR \quad\Longrightarrow\quad dP = dZ\,R^\top,\qquad dR = P^\top dZ.
$$

(Same index derivation as Linear's $dX$ and $dW$.)

### Backward, step by step (reverse order of the forward pass)

1. $Y = AV$:  $\quad dA = dY\,V^\top\;(T\times T),\qquad dV = A^\top dY\;(T\times d)$
2. $A = \operatorname{softmax}(\tilde S)$:  $\quad d\tilde S = A\odot\big(dA - \textstyle\sum_j (dA\odot A)_{:,j}\big)$  (notebook 02)
3. **Mask**: $\tilde S = S$ where allowed, and a constant $-\infty$ where masked. A constant has no dependence on $S$, so $dS = d\tilde S$ on allowed entries and $0$ on masked ones. But $A_{ij}=0$ on masked entries, so the softmax backward already produced 0 there. **Nothing extra to do.**
4. $S = QK^\top/\sqrt d$:  $\quad dQ = \frac{1}{\sqrt d}\,dS\,K,\qquad dK = \frac{1}{\sqrt d}\,dS^\top Q$
   (apply the product rule with $P=Q$, $R=K^\top$; then $d(K^\top) = Q^\top dS$, so $dK = dS^\top Q$.)

### What these mean

- $dV_j = \sum_i A_{ij}\,dY_i$. **Value $j$ collects blame from every query that read it, weighted by how much it was read.** Because of the causal mask, only queries $i \ge j$ read it. So early positions receive gradient from many later positions, and the last position's value receives gradient only from itself.
- $dQ_i = \frac{1}{\sqrt d}\sum_j dS_{ij}K_j$. The query moves toward keys whose score should have been higher ($dS_{ij}<0$ means "increase this score") and away from the others.
- $dK_j = \frac{1}{\sqrt d}\sum_i dS_{ij}Q_i$. Symmetrically, each key moves according to the queries that looked at it.

In [ ]:
def attn_single_forward(Q, K, Vv, sm):
    T, d = Q.shape
    S = Q @ K.T / np.sqrt(d)
    mask = np.triu(np.ones((T, T), dtype=bool), k=1)     # True strictly ABOVE the diagonal = future
    S = np.where(mask, -np.inf, S)
    A = sm.forward(S)
    return A @ Vv, A

def attn_single_backward(dY, Q, K, Vv, A, sm):
    d = Q.shape[1]
    dA = dY @ Vv.T
    dV = A.T @ dY
    dS = sm.backward(dA) / np.sqrt(d)                    # fold the 1/sqrt(d) in here
    dQ = dS @ K
    dK = dS.T @ Q
    return dQ, dK, dV

T, d = 4, 3
Q, K, Vv = (rng.normal(size=(T, d)) for _ in range(3))
G = rng.normal(size=(T, d))
sm = Softmax()
f = lambda: np.sum(attn_single_forward(Q, K, Vv, sm)[0] * G)

Y, A = attn_single_forward(Q, K, Vv, sm)
print("attention weights A (lower-triangular, rows sum to 1):\n", A)
dQ, dK, dV = attn_single_backward(G, Q, K, Vv, A, sm)
for name, an, arr in [("dQ", dQ, Q), ("dK", dK, K), ("dV", dV, Vv)]:
    print(name, "rel err:", rel_error(an, numerical_grad(f, arr)))

A note on the mask: `np.triu(..., k=1)` is `True` strictly above the diagonal, i.e. for key position $j > i$ (the future). We use $-\infty$. After `softmax` subtracts the row max (always finite, because the diagonal is never masked), `exp(-inf) = 0` exactly. Some code uses `-1e9` or `-1e4` instead. With float64 both are fine, but $-\infty$ makes masked weights *exactly* zero.

## 2. Why divide by $\sqrt{d}$? A gradient argument

If the entries of $q$ and $k$ are roughly independent with mean 0 and variance 1, then $q\cdot k = \sum_{m=1}^d q_m k_m$ has variance $d$. Without scaling, the scores have std $\sqrt d$ (8 for GPT-2's $d = 64$, which is the size the demo below uses). Large scores make the softmax nearly **one-hot**. And from notebook 02, $dS = A\odot(dA - \ldots)$: when $A$ is one-hot, its entries are 0 or 1, and the gradient through the softmax nearly vanishes. Dividing by $\sqrt d$ brings the score variance back to 1, keeping the softmax soft and the gradients alive.

In [ ]:
d = 64
q = rng.normal(size=(1, d)); k = rng.normal(size=(16, d))
sm_ = Softmax()
for name, scale in [("unscaled", 1.0), ("scaled 1/sqrt(d)", 1 / np.sqrt(d))]:
    a = sm_.forward(q @ k.T * scale)
    g = sm_.backward(rng.normal(size=a.shape))
    print(f"{name:18} max weight {a.max():.3f}   |grad through softmax| = {np.linalg.norm(g):.4f}")

## 3. Many heads, many sequences

### 3.1 Shapes

Input `x` is `(B, T, 768)`. With `H = 12` heads of size `hs = 768 // 12 = 64`:

1. `qkv = x @ W_attn + b_attn` gives `(B, T, 2304)`. GPT-2 uses **one fused Linear** `c_attn: C → 3C` instead of three separate ones. It's mathematically identical to three Linears (just concatenated weights), but one big matmul is faster.
2. `q, k, v = split(qkv, 3, axis=-1)` gives three `(B, T, 768)` arrays.
3. `q.reshape(B, T, 12, 64)` **splits each token's 768 features into 12 chunks of 64.** Features 0–63 belong to head 0, 64–127 to head 1, and so on. It's only a reinterpretation of the same memory.
4. `.transpose(0, 2, 1, 3)` gives `(B, 12, T, 64)`. This moves the head axis **next to the batch axis**. Now `@` multiplies the last two axes `(T, hs)` and treats `(B, H)` as batch axes, so each (sequence, head) pair gets its own independent `T×T` attention.
5. Attention, as in section 1: `(B, 12, T, T)` weights and `(B, 12, T, 64)` outputs.
6. `.transpose(0, 2, 1, 3).reshape(B, T, 768)` **concatenates the 12 heads back** per token.
7. `c_proj: 768 → 768` mixes the heads' outputs.

### 3.2 The bug in your script

Your code reshapes to `(B, T, H, hs)` but **skips step 4**, then transposes the last two axes of keys:

In [ ]:
B, T, C, H = 2, 5, 768, 12          # GPT-2 small: C = 768, H = 12
hs = C // H                          # 64
q4 = rng.normal(size=(B, T, H, hs))
k4 = rng.normal(size=(B, T, H, hs))
wrong = q4 @ np.transpose(k4, axes=[0, 1, 3, 2])
print("your scores shape:", wrong.shape, " <- (B, T, H, H): heads attend to HEADS, within one token")
q_ = q4.transpose(0, 2, 1, 3); k_ = k4.transpose(0, 2, 1, 3)
right = q_ @ k_.transpose(0, 1, 3, 2)
print("correct shape    :", right.shape, " <- (B, H, T, T): positions attend to positions, per head")

With `(B, T, H, hs)`, the last two axes are `(H, hs)`, so the matmul computes an `H×H` matrix **inside each token**: no information moves between positions at all, and a causal `(T,T)` mask can't even be applied. The NumPy code runs without error only because `H × H` broadcasts with nothing. It's a silent bug, which is exactly why gradient checks and shape assertions are worth having. Two smaller issues are also in the script: `final_projection` should be `Linear(dim_out, dim_out)` (its input is the concatenated heads, of size `dim_out`), and GPT-2's `c_attn`/`c_proj` *do* have biases.

### 3.3 Backward of the bookkeeping operations (rule 4)

| forward | backward |
|---|---|
| `np.split(qkv, 3, -1)` | `np.concatenate([dq, dk, dv], -1)` |
| `.reshape(B,T,H,hs)` | `.reshape(B,T,C)` |
| `.transpose(0,2,1,3)` | `.transpose(0,2,1,3)` (swapping axes 1↔2 is its own inverse) |

None of these change any values. They only move numbers around, so the gradient just has to be moved back to the same places.

### 3.4 The batched attention backward

Exactly section 1, with `@` batching over `(B, H)` and `.T` replaced by `.transpose(0, 1, 3, 2)` (swap only the last two axes):

```python
dA   = dy @ v.transpose(0,1,3,2)          # (B,H,T,hs) @ (B,H,hs,T) -> (B,H,T,T)
dv   = A.transpose(0,1,3,2) @ dy          # (B,H,T,T)  @ (B,H,T,hs) -> (B,H,T,hs)
datt = softmax.backward(dA) * scale       # (B,H,T,T)
dq   = datt @ k                           # (B,H,T,hs)
dk   = datt.transpose(0,1,3,2) @ q        # (B,H,T,hs)
```

In [ ]:
class CausalSelfAttention:
    def __init__(self, C, n_head, n_layer, rng, dtype=np.float64):
        assert C % n_head == 0
        self.C, self.H = C, n_head
        self.c_attn = Linear(C, 3 * C, rng=rng, dtype=dtype)                      # fused q,k,v
        self.c_proj = Linear(C, C, std=0.02 / math.sqrt(2 * n_layer), rng=rng, dtype=dtype)  # residual proj
        self.softmax = Softmax()

    def forward(self, x):
        B, T, C = x.shape
        H, hs = self.H, C // self.H
        qkv = self.c_attn.forward(x)                                   # (B, T, 3C)
        q, k, v = np.split(qkv, 3, axis=-1)                            # 3 x (B, T, C)
        q = q.reshape(B, T, H, hs).transpose(0, 2, 1, 3)               # (B, H, T, hs)
        k = k.reshape(B, T, H, hs).transpose(0, 2, 1, 3)
        v = v.reshape(B, T, H, hs).transpose(0, 2, 1, 3)
        self.scale = 1.0 / math.sqrt(hs)                               # 1/8 in GPT-2 small
        att = (q @ k.transpose(0, 1, 3, 2)) * self.scale               # (B, H, T, T)
        mask = np.triu(np.ones((T, T), dtype=bool), k=1)               # (T, T), broadcasts over B, H
        att = np.where(mask, -np.inf, att)
        A = self.softmax.forward(att)                                  # (B, H, T, T)
        y = A @ v                                                      # (B, H, T, hs)
        y = y.transpose(0, 2, 1, 3).reshape(B, T, C)                   # (B, T, C)  heads concatenated
        self.q, self.k, self.v, self.A = q, k, v, A                    # cache
        return self.c_proj.forward(y)                                  # (B, T, C)

    def backward(self, dout):                                          # dout: (B, T, C)
        B, T, C = dout.shape
        H, hs = self.H, C // self.H
        dy = self.c_proj.backward(dout)                                # (B, T, C)
        dy = dy.reshape(B, T, H, hs).transpose(0, 2, 1, 3)            # (B, H, T, hs)  undo the merge
        dA = dy @ self.v.transpose(0, 1, 3, 2)                         # (B, H, T, T)
        dv = self.A.transpose(0, 1, 3, 2) @ dy                         # (B, H, T, hs)
        datt = self.softmax.backward(dA) * self.scale                  # (B, H, T, T)  mask: zeros already
        dq = datt @ self.k                                             # (B, H, T, hs)
        dk = datt.transpose(0, 1, 3, 2) @ self.q                       # (B, H, T, hs)
        merge = lambda t: t.transpose(0, 2, 1, 3).reshape(B, T, C)     # undo the head split
        dqkv = np.concatenate([merge(dq), merge(dk), merge(dv)], axis=-1)   # (B, T, 3C)  undo split
        return self.c_attn.backward(dqkv)                              # (B, T, C)

    def parameters(self, prefix=""):
        return self.c_attn.parameters(prefix + "c_attn.") + self.c_proj.parameters(prefix + "c_proj.")

### 3.5 Full gradient check

We use larger-than-normal weights so the attention pattern is not nearly uniform. (With 0.02-scale weights the scores are almost 0 and the softmax is nearly flat, which would test less of the code.)

In [ ]:
B, T, C, H = 2, 4, 6, 2
attn = CausalSelfAttention(C, H, n_layer=1, rng=rng)
for _, p, _ in attn.parameters():
    p[...] = rng.normal(size=p.shape) * 0.7
x = rng.normal(size=(B, T, C))
G = rng.normal(size=(B, T, C))
f = lambda: np.sum(attn.forward(x) * G)

f()
dx = attn.backward(G)
print("dx rel err:", rel_error(dx, numerical_grad(f, x)))
for name, p, g in attn.parameters():
    print(f"d{name:9} rel err: {rel_error(g, numerical_grad(f, p)):.2e}")

### 3.6 Spot check at GPT-2 small's size

Now the real configuration: C = 768, 12 heads of 64, GPT-2 initialization (std 0.02, and 0.02/√24 for `c_proj`), on a sequence of T = 8 tokens.

In [ ]:
attn768 = CausalSelfAttention(768, n_head=12, n_layer=12, rng=rng)
print(f"attention parameters: {sum(p.size for _, p, _ in attn768.parameters()):,}")
x = rng.normal(size=(2, 8, 768))
G = rng.normal(size=(2, 8, 768))
f = lambda: np.sum(attn768.forward(x) * G)
f()
dx = attn768.backward(G)
print("attention weights shape:", attn768.A.shape, "  <- (B, 12 heads, T, T)")
print("dx spot rel err:", spot_check(f, x, dx))
for name, p, g in attn768.parameters():
    print(f"d{name:9} spot rel err: {spot_check(f, p, g):.2e}")

### 3.7 Testing causality (at full size)

Two tests verify the mask from opposite directions:

- **Forward:** changing the token at position `t` must not change any output at positions `< t`.
- **Backward:** a gradient placed only on the output at position `t` must not reach any input at positions `> t`.

In [ ]:
out1 = attn768.forward(x).copy()
x2 = x.copy(); x2[:, 5, :] += 5.0                   # perturb position 5 only (of 0..7)
out2 = attn768.forward(x2)
print("outputs changed at positions:", np.where(np.abs(out1 - out2).max(axis=(0, 2)) > 1e-12)[0])

attn768.forward(x)
G_pos2 = np.zeros_like(G); G_pos2[:, 2, :] = 1.0    # gradient only on output position 2
dx = attn768.backward(G_pos2)
print("inputs receiving gradient   :", np.where(np.abs(dx).max(axis=(0, 2)) > 1e-12)[0])

## 4. Notes

- **Padding masks.** GPT-2 training cuts fixed-length chunks from one continuous token stream, so there is no padding and no padding mask. You only need one for variable-length batches, e.g. fine-tuning on separate documents. Your `padding_mask` argument can go for now.
- **Cost.** The `(B, 12, T, T)` attention matrix is the memory bottleneck: it grows with $T^2$. At T = 1024 and B = 8 it is 8 × 12 × 1024 × 1024 floats = 403 MB per layer in float32, and the backward pass needs it for all 12 layers. FlashAttention avoids storing it by recomputing it in tiles during the backward pass. Not needed here.
- **Dropout.** GPT-2 used dropout 0.1 on attention weights and residual outputs. For learning, and for small from-scratch runs, leave it out. (Its backward pass is just multiplying by the same random mask.)

## Summary

```
forward : x -> c_attn -> split -> reshape/transpose -> QK^T/sqrt(hs) -> mask -> softmax -> @V -> merge heads -> c_proj
backward: run every arrow in reverse; matmul gives two gradients; mask needs nothing; bookkeeping ops invert.
```

Next: **05 — Embeddings, blocks and the full GPT**, where residual connections and weight tying meet, and we gradient-check the entire model.